# 06 - Manuscript-Final Results (ARIMA and Seasonal-Naive excluded)

This notebook reuses every output already computed by Notebooks 04-05 --
no models are retrained and no predictions are regenerated -- and rebuilds
every table, significance test summary, and figure with **ARIMA and
Seasonal-Naive excluded**.

**Why these two, specifically, and not others:** both recorded test MSE
roughly 200x larger than every other model at every horizon (Table 3 /
`test_metrics_full.csv`), a scale gap wide enough that including them
flattens every other model's comparison to visual/numerical noise and adds
no discriminating information among the nine models that ARE comparable.
This is a reporting decision, not a re-analysis -- their full results remain
available in Notebook 05's unfiltered output for anyone who wants them
(nothing is deleted, only excluded from the manuscript-facing package built
here).

**Kept in this comparison:** Naive (persistence), LSTM, TCN, N-BEATS,
DLinear, NLinear, PatchTST, iTransformer, ELinear -- all nine were within a
comparable order of magnitude of each other at every horizon.

## Colab setup
Run this once per session. Mounts Google Drive, `cd`s into `notebooks/`,
installs nothing extra (this notebook only reads CSV/JSON outputs already
produced by Notebooks 04-05 -- no model training, no GPU needed).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/potato_pipeline"
os.chdir(os.path.join(PROJECT_ROOT, "notebooks"))
print("Working directory:", os.getcwd())

In [ ]:
import sys, os, json, shutil
sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = "../results"
FIG_DIR = "../results/figures_manuscript"       # SEPARATE from Notebook 05's figures/ -- nothing overwritten
TAB_DIR = "../results/tables_manuscript"         # SEPARATE from Notebook 05's tables/
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

HORIZONS = [1, 7, 14, 30]
EXCLUDED_MODELS = ["ARIMA", "SeasonalNaive"]

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
})

print(f"Excluding from every table/test/figure in this notebook: {EXCLUDED_MODELS}")

## 1. Load and filter every upstream result file

Each file is loaded once here; every table and figure below reuses these already-filtered dataframes, so the exclusion can never be applied inconsistently between sections.

In [ ]:
metrics_table = pd.read_csv(f"{RESULTS_DIR}/test_metrics_full.csv")
metrics_table = metrics_table[~metrics_table["model"].isin(EXCLUDED_MODELS)].copy()

dm_table = pd.read_csv(f"{RESULTS_DIR}/dm_significance_tests.csv")
dm_table = dm_table[~dm_table["benchmark"].isin(EXCLUDED_MODELS)].copy()

dm_pairwise_path = f"{RESULTS_DIR}/dm_pairwise_full.csv"
dm_pairwise = None
if os.path.exists(dm_pairwise_path):
    dm_pairwise = pd.read_csv(dm_pairwise_path)
    dm_pairwise = dm_pairwise[
        (~dm_pairwise["model_a"].isin(EXCLUDED_MODELS)) &
        (~dm_pairwise["model_b"].isin(EXCLUDED_MODELS))
    ].copy()

with open(f"{RESULTS_DIR}/param_counts_by_horizon.json") as f:
    param_counts_by_horizon = json.load(f)
param_counts_by_horizon = {k: v for k, v in param_counts_by_horizon.items() if k not in EXCLUDED_MODELS}
# (ARIMA/Naive/SeasonalNaive were never in this file anyway -- only DL models have trainable parameters)

print(f"metrics_table: {metrics_table['model'].nunique()} models remain -> {sorted(metrics_table['model'].unique())}")
print(f"dm_table: {dm_table['benchmark'].nunique()} benchmarks remain -> {sorted(dm_table['benchmark'].unique())}")

## 2. Main results table (MSE, MAPE by model x horizon)

Same format as Notebook 05 Table 3, with ARIMA and Seasonal-Naive removed.
Lowest error per horizon is bolded among the nine remaining models.

In [ ]:
def build_main_table(metrics_table, metric="MSE"):
    pivot_mean = metrics_table.pivot(index="model", columns="horizon", values=f"{metric}_mean")
    pivot_std = metrics_table.pivot(index="model", columns="horizon", values=f"{metric}_std")

    formatted = pivot_mean.copy().astype(object)
    for col in pivot_mean.columns:
        best_idx = pivot_mean[col].idxmin()
        for idx in pivot_mean.index:
            mean_v, std_v = pivot_mean.loc[idx, col], pivot_std.loc[idx, col]
            cell = f"{mean_v:.2f} ({std_v:.2f})"
            if idx == best_idx:
                cell = f"**{cell}**"
            formatted.loc[idx, col] = cell
    formatted.columns = [f"h={c}" for c in formatted.columns]
    return formatted

for metric in ["MSE", "RMSE", "MAE", "MAPE", "sMAPE"]:
    tbl = build_main_table(metrics_table, metric)
    tbl.to_csv(f"{TAB_DIR}/manuscript_results_{metric.lower()}.csv")

mse_table = build_main_table(metrics_table, "MSE")
mse_table

## 3. Diebold-Mariano significance -- ELinear vs. each remaining benchmark

In [ ]:
dm_summary = dm_table.pivot(index="benchmark", columns="horizon", values="elinear_significantly_better")
dm_pvals = dm_table.pivot(index="benchmark", columns="horizon", values="p_value_holm")

dm_display = dm_summary.copy().astype(object)
for col in dm_summary.columns:
    for idx in dm_summary.index:
        sig = dm_summary.loc[idx, col]
        p = dm_pvals.loc[idx, col]
        marker = "check (p={:.3f})".format(p) if sig else "n.s. (p={:.3f})".format(p)
        dm_display.loc[idx, col] = marker
dm_display.columns = [f"h={c}" for c in dm_display.columns]
dm_display.to_csv(f"{TAB_DIR}/manuscript_dm_significance_summary.csv")
dm_display

## 4. Pairwise DM significance heatmap, all remaining models

Full all-vs-all comparison (not just ELinear-vs-benchmark), same 2x2
per-horizon layout as Notebook 05, with ARIMA/Seasonal-Naive removed from
both axes.

In [ ]:
if dm_pairwise is not None:
    model_order = ["Naive", "LSTM", "TCN", "NBEATS", "DLinear", "NLinear", "PatchTST", "iTransformer", "ELinear"]

    fig, axes = plt.subplots(2, 2, figsize=(13, 12))
    for ax, h in zip(axes.flat, HORIZONS):
        sub = dm_pairwise[dm_pairwise.horizon == h]
        models_h = [m for m in model_order if m in set(sub.model_a) | set(sub.model_b)]
        n = len(models_h)

        # Lookup by unordered pair -- one row per pair in dm_pairwise_full.csv,
        # so the same (dm_stat, p_value_holm) is shown at both [i,j] and [j,i];
        # color alone (not text sign) carries which model was favored.
        pair_lookup = {}
        for _, row in sub.iterrows():
            pair_lookup[frozenset((row.model_a, row.model_b))] = row

        matrix = np.zeros((n, n))
        for _, row in sub.iterrows():
            if not row.significant:
                continue
            i, j = models_h.index(row.model_a), models_h.index(row.model_b)
            val = 1 if row.better_model == row.model_a else -1
            matrix[i, j] = val
            matrix[j, i] = -val

        im = ax.imshow(matrix, cmap="RdBu", vmin=-1, vmax=1)
        ax.set_xticks(range(n)); ax.set_xticklabels(models_h, rotation=90, fontsize=8)
        ax.set_yticks(range(n)); ax.set_yticklabels(models_h, fontsize=8)
        ax.set_title(f"h = {h} d")

        # Annotate every off-diagonal cell with the DM statistic and
        # Holm-corrected p-value for that pair.
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                row = pair_lookup.get(frozenset((models_h[i], models_h[j])))
                if row is None:
                    continue
                text = f"{row.dm_stat:.2f}\n(p={row.p_value_holm:.3f})"
                text_color = "white" if abs(matrix[i, j]) > 0.5 else "black"
                ax.text(j, i, text, ha="center", va="center", fontsize=5.5, color=text_color)

    fig.suptitle("Pairwise DM significance, ARIMA and Seasonal-Naive excluded\n"
                 "(blue = row model significantly better, red = column model significantly better; "
                 "cell text = DM statistic and Holm-corrected p-value)", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/dm_pairwise_heatmap_manuscript.pdf", bbox_inches="tight")
    plt.savefig(f"{FIG_DIR}/dm_pairwise_heatmap_manuscript.png", bbox_inches="tight")
    plt.show()
else:
    print("dm_pairwise_full.csv not found -- skipping full pairwise heatmap (Section 3's ELinear-vs-benchmark table above still applies).")

## 5. Figure: Error vs. forecast horizon, remaining models only

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for model_name, grp in metrics_table.groupby("model"):
    grp = grp.sort_values("horizon")
    is_elinear = (model_name == "ELinear")
    ax.plot(grp["horizon"], grp["MSE_mean"], marker="o", label=model_name,
            lw=2.0 if is_elinear else 1.2, zorder=10 if is_elinear else 2)
    ax.fill_between(grp["horizon"], grp["MSE_mean"] - grp["MSE_std"], grp["MSE_mean"] + grp["MSE_std"], alpha=0.12)

ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Test MSE")
ax.set_title("Agra Potato Price: Forecast Error vs. Horizon\n(ARIMA and Seasonal-Naive excluded)")
ax.set_xticks(HORIZONS)
ax.legend(loc="upper left", fontsize=8, ncol=2, frameon=False)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/error_vs_horizon_manuscript.pdf", bbox_inches="tight")
plt.savefig(f"{FIG_DIR}/error_vs_horizon_manuscript.png", bbox_inches="tight")
plt.show()

## 6. Figure: Prediction vs. actual, all horizons

Identical construction to Notebook 05 Section 4 (PLOT_MODELS there already
excluded ARIMA/Seasonal-Naive for the same scale reason) -- reproduced here
so this notebook is a complete, standalone source for every manuscript
figure without needing to cross-reference Notebook 05.

In [ ]:
PLOT_MODELS = ["Naive", "LSTM", "TCN", "NBEATS", "DLinear", "NLinear", "PatchTST", "iTransformer", "ELinear"]
WONG_PALETTE = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#999999", "#8B0000"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=False)
legend_handles, legend_labels = None, None

for ax, h in zip(axes.flat, HORIZONS):
    path = f"{RESULTS_DIR}/pred_vs_actual_h{h}.csv"
    if not os.path.exists(path):
        ax.set_title(f"h = {h} d (no data -- run Notebook 04's export step first)")
        continue

    df_h = pd.read_csv(path, parse_dates=["date"])
    actual_line, = ax.plot(df_h["date"], df_h["actual"], label="Actual", color="black", lw=2.0, zorder=100)
    handles = [actual_line]
    for model_name, color in zip(PLOT_MODELS, WONG_PALETTE):
        if model_name not in df_h.columns:
            continue
        is_elinear = (model_name == "ELinear")
        line, = ax.plot(
            df_h["date"], df_h[model_name], label=model_name,
            color=("black" if is_elinear else color),
            lw=(2.2 if is_elinear else 1.1),
            linestyle=("--" if is_elinear else "-"),
            alpha=(0.95 if is_elinear else 0.8),
            zorder=(90 if is_elinear else 10),
        )
        handles.append(line)

    ax.set_xlabel("Date"); ax.set_ylabel("Rs/quintal"); ax.set_title(f"h = {h} d")
    ax.tick_params(axis="x", rotation=30)
    if legend_handles is None:
        legend_handles = handles
        legend_labels = [h_.get_label() for h_ in handles]

fig.suptitle("Agra Potato Price: Forecast vs. Actual, by Horizon", y=1.02)
fig.legend(legend_handles, legend_labels, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/pred_vs_actual_all_horizons_manuscript.pdf", bbox_inches="tight")
plt.savefig(f"{FIG_DIR}/pred_vs_actual_all_horizons_manuscript.png", bbox_inches="tight")
plt.show()

## 7. Figure: Model complexity vs. accuracy, all horizons

No change needed from Notebook 05 -- ARIMA/Naive/Seasonal-Naive were never
part of `param_counts_by_horizon.json` in the first place (they have no
trainable parameters), so this figure was already exclusive to the nine
comparable models. Reproduced here for a self-contained notebook.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, h in zip(axes.flat, HORIZONS):
    subset = metrics_table[metrics_table["horizon"] == h].copy()
    param_map = {m: counts.get(str(h)) for m, counts in param_counts_by_horizon.items()}
    subset["params"] = subset["model"].map(param_map)
    subset = subset.dropna(subset=["params"])

    if len(subset) > 0:
        ax.scatter(subset["params"], subset["MSE_mean"])
        for _, row in subset.iterrows():
            ax.annotate(row["model"], (row["params"], row["MSE_mean"]), fontsize=7,
                         xytext=(4, 4), textcoords="offset points")
        ax.set_xscale("log")
        ax.set_xlabel("Parameter count (log scale)")
        ax.set_ylabel("Test MSE")
        ax.set_title(f"h = {h} d")
    else:
        ax.set_title(f"h = {h} d (no data)")

fig.suptitle("Model Complexity vs. Accuracy, by Horizon")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/complexity_vs_accuracy_manuscript.pdf", bbox_inches="tight")
plt.savefig(f"{FIG_DIR}/complexity_vs_accuracy_manuscript.png", bbox_inches="tight")
plt.show()

## 8. Figure: Directional accuracy (1-day horizon), remaining models only

In [ ]:
dir_acc = metrics_table[metrics_table["horizon"] == 1][["model", "DirAcc_mean", "DirAcc_std"]].sort_values("DirAcc_mean", ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(dir_acc["model"], dir_acc["DirAcc_mean"], yerr=dir_acc["DirAcc_std"], capsize=3)
ax.axhline(50, color="grey", ls="--", lw=1, label="Chance level (50%)")
ax.set_ylabel("Directional accuracy (%)")
ax.set_title("1-day-ahead Directional Accuracy\n(ARIMA and Seasonal-Naive excluded)")
plt.xticks(rotation=45, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/directional_accuracy_h1_manuscript.pdf", bbox_inches="tight")
plt.show()

## 9. Export the manuscript-ready package

Separate folder from Notebook 05's full (unfiltered) package -- nothing
from the complete technical record is overwritten.

In [ ]:
package_dir = f"{RESULTS_DIR}/manuscript_package_final"
os.makedirs(package_dir, exist_ok=True)
for src_dir in [TAB_DIR, FIG_DIR]:
    for fname in os.listdir(src_dir):
        shutil.copy(os.path.join(src_dir, fname), os.path.join(package_dir, fname))

metrics_table.to_csv(f"{package_dir}/test_metrics_filtered.csv", index=False)
dm_table.to_csv(f"{package_dir}/dm_significance_filtered.csv", index=False)

print(f"Manuscript-final package assembled at: {package_dir}")
print(f"Models included: {sorted(metrics_table['model'].unique())}")
print(f"Models excluded: {EXCLUDED_MODELS}")
print(os.listdir(package_dir))